# 01 · Build multimodal artifacts (Kaggle GPU)

This notebook does the **heavy** part of the pipeline so your laptop doesn't have to:

1. download MMLongBench-Doc and pick the 20-document subset
2. extract text, tables and figures (PyMuPDF), and OCR the scanned pages (PaddleOCR)
3. OCR chart labels, generate **BLIP** captions for every figure
4. embed every chunk with **bge-small** (text) and every figure with **CLIP** (images)
5. zip everything into **`artifacts.zip`**, then download it from the *Output* tab

**Before running:** Settings → Accelerator **GPU T4 x1** (or P100) · Internet **ON**.
Runtime about 25 to 40 min for 20 documents.

In [ ]:
# repository and branch to run
REPO_URL = "https://github.com/yayyyyshi/multimodal_rag.git"
BRANCH   = "main"  # use "feature/full-pipeline" until the PR is merged
import os, subprocess
os.chdir("/kaggle/working")
if not os.path.exists("multimodal_rag"):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL], check=True)
os.chdir("/kaggle/working/multimodal_rag")
!git log --oneline -3

In [ ]:
!pip install -q -r requirements-kaggle.txt 2>&1 | tail -3
import torch; print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 1 · Dataset subset (the benchmark is cloned to /kaggle/temp so it is not part of the output)

In [ ]:
!python scripts/prepare_dataset.py --local-dir /kaggle/temp/mmlongbench

## 2 · Extraction + OCR of scanned pages + chunking

In [ ]:
!python scripts/ingest.py --subset

## 3 · Figure OCR + BLIP captions + text/CLIP embeddings → artifacts.zip

In [ ]:
!python scripts/build_embeddings.py --ocr-figures --zip

## 4 · Sanity check: build the index here and look at retrieval quality

In [ ]:
!python scripts/build_index.py
!python scripts/evaluate.py --retrieval-only --run-name kaggle_retrieval

In [ ]:
# first five figure descriptions (BLIP caption and OCR text)
import json
rows = [json.loads(l) for l in open("artifacts/chunks.jsonl")]
for r in [r for r in rows if r["modality"] == "image"][:5]:
    print(r["chunk_id"]); print(r["content"][:400]); print("-" * 80)

## 5 · Download
Open the **Output** panel (right side) → `multimodal_rag/artifacts.zip` → download.
On your laptop, put it in the project folder and run:

```bash
python scripts/build_index.py --zip artifacts.zip
```
Tip: also click **Save Version → Save & Run All** so the output is kept and notebook 02 can use it as input.